# 02: Dataset Preprocessing & Data Leakage Prevention

## 1. Research Objective
To preprocess the SuicideWatch dataset, mitigating data leakage, handling missing data, and generating a reproducible, stratified Train/Validation/Test split while preserving meaningful linguistic features.

## 2. Input
Raw SuicideWatch dataset file (CSV) located securely in Google Drive.

## 3. Method
1. **Leakage Prevention**: Elimination of duplicates and conflicting labels.
2. **Text Cleaning**: Light URL removal and HTML decoding, preserving punctuation, hashtags, emojis, and negations.
3. **Data Splitting**: Stratified splitting to maintain class proportions across partitions.

## 4. Output
- `data/processed/suicide_watch_cleaned_split.csv`: A single preprocessed file containing a new `split` column (train, val, test) to avoid unnecessary file duplication.
- `results/metrics/split_metrics.json`: Metadata summarizing the split proportions.

## 5. Experimental Decisions
- Fixed random seed for reproducible splitting.
- Strict deduplication across target labels to prevent bleeding between Train and Test sets.
- Configurable dataset path.

## 6. Evaluation Metrics
Dataset volume retention (before vs. after cleaning) and class distribution balance.

In [ ]:
import sys
import os
import yaml
import json
import pandas as pd

# 1. Setup Environment
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/ML_Dataset'
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        PROJECT_ROOT = '../'
    else:
        PROJECT_ROOT = '.'

sys.path.append(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f"Working directory set to: {os.getcwd()}")

from src.preprocessing import clean_social_media_text, remove_data_leakage_and_noise, create_stratified_splits

# Load Config for Seed and Data Path
config_path = "configs/experiment.yaml"
seed = 42
data_path = "data/"
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config = yaml.safe_load(f)
        seed = config.get("seed", 42)
        data_path = config.get("data", {}).get("path", "data/")

DATASET_PATH = os.path.join(data_path, "Suicide_Detection.csv")
print(f"Using fixed random seed: {seed}")
print(f"Expected Dataset Path: {DATASET_PATH}")

## Load Dataset & Identify Columns
Load the raw data and establish the specific column names for text and labels dynamically (based on findings from Phase 1).

In [ ]:
if not os.path.exists(DATASET_PATH):
    print(f"[WARNING] Dataset not found at {DATASET_PATH}. Please provide the correct absolute path.")
else:
    df = pd.read_csv(DATASET_PATH)
    print(f"Raw dataset loaded. Size: {len(df)} rows")
    
    # Dynamically identify text and label columns
    text_col = 'text' if 'text' in df.columns else df.columns[0]
    label_col = 'class' if 'class' in df.columns else df.columns[1]
    
    print(f"Using Text Column: '{text_col}'")
    print(f"Using Label Column: '{label_col}'")

## Noise Removal & Data Leakage Prevention
Executes deduplication protocols from `src.preprocessing` to eliminate texts with conflicting labels, preventing train/test contamination.

In [ ]:
if 'df' in locals() and not df.empty:
    print(f"Rows before deduplication/leakage prevention: {len(df)}")
    
    # Remove duplicates, empty texts, and conflicting cross-label items
    df_clean = remove_data_leakage_and_noise(df, text_col, label_col)
    
    dropped_rows = len(df) - len(df_clean)
    print(f"Rows after deduplication/leakage prevention: {len(df_clean)}")
    print(f"Total invalid/duplicate rows removed: {dropped_rows}")

## Text Cleaning
Applies a light linguistic clean. 
**Note:** Aggressive normalization (like removing emojis, punctuation, or negations) is intentionally avoided to preserve the emotional context of the post, which is highly predictive of distress.

In [ ]:
if 'df_clean' in locals():
    print("Cleaning social media text...")
    # Apply cleaning function from src.preprocessing
    df_clean[text_col] = df_clean[text_col].apply(clean_social_media_text)
    
    # Final drop of any rows that became empty after cleaning (e.g. posts that were only a URL)
    df_clean = df_clean[df_clean[text_col].astype(str).str.strip() != ""]
    
    print(f"Final clean dataset size: {len(df_clean)}")

## Stratified Data Splitting
Creates Train (80%), Validation (10%), and Test (10%) splits using a fixed random seed. Instead of saving three duplicate large files, we append a `split` column to the original dataset and save it once.

In [ ]:
if 'df_clean' in locals() and not df_clean.empty:
    # Split ratios
    train_ratio = 0.8
    val_ratio = 0.1
    test_ratio = 0.1
    
    df_split = create_stratified_splits(
        df=df_clean, 
        label_col=label_col, 
        train_size=train_ratio, 
        val_size=val_ratio, 
        test_size=test_ratio, 
        seed=seed
    )
    
    # Extract distributions for the metrics report
    train_dist = df_split[df_split['split'] == 'train'][label_col].value_counts().to_dict()
    val_dist = df_split[df_split['split'] == 'val'][label_col].value_counts().to_dict()
    test_dist = df_split[df_split['split'] == 'test'][label_col].value_counts().to_dict()
    
    print("--- Split Sizes ---")
    print(f"Train: {sum(train_dist.values())}")
    print(f"Validation: {sum(val_dist.values())}")
    print(f"Test: {sum(test_dist.values())}")
    
    print("\n--- Class Distribution (Train) ---")
    for k, v in train_dist.items():
        print(f"{k}: {v}")
    
    # Save split metadata to metrics
    split_metrics = {
        "seed_used": seed,
        "train_ratio": train_ratio,
        "val_ratio": val_ratio,
        "test_ratio": test_ratio,
        "train_size": sum(train_dist.values()),
        "val_size": sum(val_dist.values()),
        "test_size": sum(test_dist.values()),
        "train_distribution": train_dist,
        "val_distribution": val_dist,
        "test_distribution": test_dist
    }
    
    os.makedirs('results/metrics', exist_ok=True)
    with open('results/metrics/split_metrics.json', 'w') as f:
        json.dump(split_metrics, f, indent=4)
    
    # Save unified split dataset
    os.makedirs('data/processed', exist_ok=True)
    processed_path = 'data/processed/suicide_watch_cleaned_split.csv'
    df_split.to_csv(processed_path, index=False)
    
    print(f"\nProcessed dataset saved to: {processed_path}")
    print("Split metrics saved to: results/metrics/split_metrics.json")

## Preprocessing Methodology Implications

**Preprocessing Pipeline**:
The data cleaning pipeline specifically avoided aggressive normalizations (such as removing all non-alphanumeric characters, stop words, or emojis) because in the context of distress or suicidal ideation detection, linguistic nuance (like repeating punctuation `...` or `???` and negation words like `not`, `never`) carries immense psychological signal. Only unhelpful artifacts like URLs and HTML entities were stripped.

**Leakage Prevention**:
We removed identical text sequences that mapped to different classification labels (conflicting data), and kept only one copy of identical text sequences with identical labels (standard duplicates). This prevents a duplicated post from appearing in both the training set and the test set simultaneously, artificially inflating model evaluation metrics.

**Split Strategy**:
A standard 80% / 10% / 10% Train-Validation-Test split was utilized. The split was stratified by the target label (`class`) to ensure an equal distribution of distress vs. non-distress posts across all partitions. By saving the result into a new `split` column rather than splitting files entirely, we avoid generating redundant bulky CSV files and simplify tracking.

**Limitations**:
- Light preprocessing retains spelling errors and internet slang, which standard transformers (like BERT or RoBERTa) will need to handle via subword tokenization.
- Texts that were entirely URL-based or purely HTML artifacts were entirely discarded, slightly altering the absolute dataset volume from Phase 1.